<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/Embrapa_VGG16_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()  # Upload kaggle.json

!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset
!kaggle datasets download -d poornimasinghthakur/embrapa
!unzip -q embrapa.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/poornimasinghthakur/embrapa
License(s): unknown
 97% 1.02G/1.05G [00:11<00:00, 66.6MB/s]
100% 1.05G/1.05G [00:11<00:00, 96.6MB/s]


In [ ]:
# ===========================
# 2. Imports & Data Pipeline
# ===========================
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing import image_dataset_from_directory

img_size = (224, 224)  # VGG16 input size
batch_size = 32

# Load once without augmentation to get class names
tmp_train_ds = image_dataset_from_directory(
    "/content/XDB/train",
    image_size=img_size,
    batch_size=batch_size
)
class_names = tmp_train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

# Augment training data
train_ds = tmp_train_ds.map(lambda x, y: (data_augmentation(x), y))

# Validation & Test datasets
val_ds = image_dataset_from_directory(
    "/content/XDB/val",
    image_size=img_size,
    batch_size=batch_size
)
test_ds = image_dataset_from_directory(
    "/content/XDB/test",
    image_size=img_size,
    batch_size=batch_size
)

# Convert labels to one-hot encoding for CategoricalCrossentropy
def one_hot_map(images, labels):
    return images, tf.one_hot(labels, depth=num_classes)

train_ds_onehot = train_ds.map(one_hot_map)
val_ds_onehot = val_ds.map(one_hot_map)
test_ds_onehot = test_ds.map(one_hot_map)

# Prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds_onehot = train_ds_onehot.prefetch(AUTOTUNE)
val_ds_onehot = val_ds_onehot.prefetch(AUTOTUNE)
test_ds_onehot = test_ds_onehot.prefetch(AUTOTUNE)

Found 29607 files belonging to 93 classes.
Classes: ['AlgodтФЬ├║o (Cotton) - Mancha de Mirotecio (Myrothecium leaf spot) - Cropped', 'AlgodтФЬ├║o (Cotton) - Mela (Soreshin) - Cropped', 'AlgodтФЬ├║o (Cotton) - Ramularia (Areolate Mildew) - Cropped', 'Arroz (Rice) - Brusone (Rice Blast) - Cropped', 'Arroz (Rice) - Escaldadura (Leaf Scald) - Cropped', 'CafтФЬтМР (Coffee) - BichoMineiro (Leaf Miner) - Cropped', 'CafтФЬтМР (Coffee) - Cercosporiose (Cercospora Leaf Spot) - Cropped', 'CafтФЬтМР (Coffee) - Ferrugem (Rust) - Cropped', 'CafтФЬтМР (Coffee) - Mancha Aureolada (Bacterial Blight) - Cropped', 'CafтФЬтМР (Coffee) - Mancha Mantegosa (Blister Spot) - Cropped', 'CafтФЬтМР (Coffee) - Phoma (Brown leaf spot) - Cropped', 'Cajueiro (Cashew Tree) - Alga (Algae) - Cropped', 'Cajueiro (Cashew Tree) - Antracnose (Anthracnose) - Cropped', 'Cajueiro (Cashew Tree) - Mancha Angular (Angular Leaf Spot) - Cropped', 'Cajueiro (Cashew Tree) - Mofo Preto (Black Mould) - Cropped', 'Cajueiro (Cashew Tree) 

In [ ]:
base_model = VGG16(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation='softmax')
])

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
history = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=10
)

Epoch 1/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 523s 549ms/step - accuracy: 0.1738 - loss: 4.5234 - val_accuracy: 0.4695 - val_loss: 2.8860
Epoch 2/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 534s 533ms/step - accuracy: 0.3663 - loss: 3.1021 - val_accuracy: 0.5489 - val_loss: 2.5279
Epoch 3/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 497s 529ms/step - accuracy: 0.4155 - loss: 2.8690 - val_accuracy: 0.5881 - val_loss: 2.4167
Epoch 4/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 492s 531ms/step - accuracy: 0.4337 - loss: 2.7778 - val_accuracy: 0.5810 - val_loss: 2.3403
Epoch 5/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 490s 529ms/step - accuracy: 0.4491 - loss: 2.7258 - val_accuracy: 0.5890 - val_loss: 2.3040
Epoch 6/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 489s 528ms/step - accuracy: 0.4581 - loss: 2.6935 - val_accuracy: 0.6135 - val_loss: 2.2859
Epoch 7/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 490s 529ms/step - accuracy: 0.4623 - loss: 2.6666 - val_accuracy: 0.6113 - val_loss: 2.2297
Epoch 8/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 490s 529ms/step - accuracy: 0.4621 -

In [ ]:
base_model.trainable = True
fine_tune_at = 10
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=20,
    initial_epoch=10
)

Epoch 11/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 592s 620ms/step - accuracy: 0.5280 - loss: 2.4283 - val_accuracy: 0.7308 - val_loss: 1.8813
Epoch 12/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 567s 612ms/step - accuracy: 0.6219 - loss: 2.1729 - val_accuracy: 0.7565 - val_loss: 1.7999
Epoch 13/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 569s 614ms/step - accuracy: 0.6656 - loss: 2.0559 - val_accuracy: 0.7898 - val_loss: 1.7094
Epoch 14/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 566s 611ms/step - accuracy: 0.6925 - loss: 1.9917 - val_accuracy: 0.7958 - val_loss: 1.6754
Epoch 15/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 572s 617ms/step - accuracy: 0.7148 - loss: 1.9358 - val_accuracy: 0.8172 - val_loss: 1.6344
Epoch 16/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 619s 614ms/step - accuracy: 0.7351 - loss: 1.8849 - val_accuracy: 0.8249 - val_loss: 1.6070
Epoch 17/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 556s 600ms/step - accuracy: 0.7458 - loss: 1.8433 - val_accuracy: 0.8327 - val_loss: 1.5748
Epoch 18/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 569s 608ms/step - accuracy: 

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize

TEMPERATURE = 2.0

def predict_with_temperature(model, dataset, T=1.0):
    all_probs, all_labels = [], []
    for images, labels in dataset:
        logits = model(images, training=False)
        scaled_logits = logits / T
        probs = tf.nn.softmax(scaled_logits).numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_probs)

y_true, y_probs = predict_with_temperature(model, test_ds_onehot, T=TEMPERATURE)
y_pred = np.argmax(y_probs, axis=1)

y_true_bin = label_binarize(y_true.argmax(axis=1), classes=range(num_classes))

accuracy = accuracy_score(y_true.argmax(axis=1), y_pred)
precision = precision_score(y_true.argmax(axis=1), y_pred, average='weighted')
recall = recall_score(y_true.argmax(axis=1), y_pred, average='weighted')
f1 = f1_score(y_true.argmax(axis=1), y_pred, average='weighted')
kappa = cohen_kappa_score(y_true.argmax(axis=1), y_pred)
auc = roc_auc_score(y_true_bin, y_probs, average='macro', multi_class='ovr')

print("\n Evaluation Metrics After Calibration:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Kappa    : {kappa:.4f}")

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



 Evaluation Metrics After Calibration:
Accuracy : 0.8438
Precision: 0.8405
Recall   : 0.8438
F1 Score : 0.8289
AUC      : 0.9935
Kappa    : 0.8374
